# 01 · 数据集探索分析
**项目**: 工业表面缺陷检测 | **数据集**: NEU-DET 钢铁表面缺陷数据集

本 Notebook 完成以下工作：
1. 生成/加载数据集，查看基本统计信息
2. 可视化各类别缺陷样本
3. 分析类别分布（是否存在类别不平衡）
4. 分析目标框尺寸分布
5. 数据增强效果预览

In [ ]:
import sys
sys.path.insert(0, '..')

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from collections import Counter
import pandas as pd

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

print('环境加载完成 ✓')

## 1. 生成演示数据集

In [ ]:
from src.dataset import generate_synthetic_samples, DEFECT_CLASSES

DATA_DIR = '../data/synthetic'
generate_synthetic_samples(output_dir=DATA_DIR, n_per_class=100)
print(f'\n缺陷类别: {DEFECT_CLASSES}')

## 2. 数据集基本统计

In [ ]:
data_path = Path(DATA_DIR)
stats = {}
total = 0

print(f'{"Split":<10} {"Images":>8} {"Labels":>8}')
print('-' * 30)
for split in ['train', 'val', 'test']:
    img_dir = data_path / 'images' / split
    lbl_dir = data_path / 'labels' / split
    n_img = len(list(img_dir.glob('*.jpg'))) if img_dir.exists() else 0
    n_lbl = len(list(lbl_dir.glob('*.txt'))) if lbl_dir.exists() else 0
    stats[split] = {'images': n_img, 'labels': n_lbl}
    total += n_img
    print(f'{split:<10} {n_img:>8} {n_lbl:>8}')

print('-' * 30)
print(f'{"Total":<10} {total:>8}')

In [ ]:
# 统计各类别样本数量
class_counts = Counter()
bbox_sizes = []

for split in ['train', 'val', 'test']:
    lbl_dir = data_path / 'labels' / split
    if not lbl_dir.exists():
        continue
    for lbl_file in lbl_dir.glob('*.txt'):
        with open(lbl_file) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    cls_id = int(parts[0])
                    bw, bh = float(parts[3]), float(parts[4])
                    class_counts[DEFECT_CLASSES[cls_id]] += 1
                    bbox_sizes.append((bw * 200, bh * 200))  # 假设图像200x200

df_counts = pd.DataFrame([
    {'类别': cls, '中文名': zh, '数量': class_counts.get(cls, 0)}
    for cls, zh in zip(DEFECT_CLASSES, ['龟裂','夹杂物','斑点','麻面','压入氧化皮','划痕'])
])
df_counts['占比(%)'] = (df_counts['数量'] / df_counts['数量'].sum() * 100).round(1)
print(df_counts.to_string(index=False))

## 3. 类别分布可视化

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 柱状图
zh_names = ['龟裂','夹杂物','斑点','麻面','压入\n氧化皮','划痕']
counts = [class_counts.get(cls, 0) for cls in DEFECT_CLASSES]
colors = plt.cm.Set2(np.linspace(0, 1, len(DEFECT_CLASSES)))

bars = axes[0].bar(zh_names, counts, color=colors, edgecolor='white', linewidth=1.5)
for bar, cnt in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 str(cnt), ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_title('各类别样本数量分布', fontsize=13, fontweight='bold')
axes[0].set_ylabel('样本数量')
axes[0].set_ylim(0, max(counts) * 1.2)

# 饼图
wedges, texts, autotexts = axes[1].pie(
    counts, labels=zh_names, colors=colors,
    autopct='%1.1f%%', startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for at in autotexts:
    at.set_fontsize(9)
axes[1].set_title('各类别占比', fontsize=13, fontweight='bold')

plt.suptitle('NEU-DET 数据集类别分布分析', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../results/demo/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ 已保存: results/demo/class_distribution.png')

## 4. 各类别样本可视化

In [ ]:
zh_names_map = dict(zip(DEFECT_CLASSES, ['龟裂','夹杂物','斑点','麻面','压入氧化皮','划痕']))
COLOR_MAP = {
    'crazing': '#00FF00', 'inclusion': '#FF0000', 'patches': '#0000FF',
    'pitted_surface': '#FFFF00', 'rolled-in_scale': '#FF00FF', 'scratches': '#00FFFF'
}

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('各类别缺陷样本示例（含标注框）', fontsize=16, fontweight='bold')

for ax, cls_name in zip(axes.flat, DEFECT_CLASSES):
    # 寻找该类别的第一张图
    found = False
    for split in ['train', 'val', 'test']:
        img_dir = data_path / 'images' / split
        lbl_dir = data_path / 'labels' / split
        candidates = list(img_dir.glob(f'{cls_name}_*.jpg')) if img_dir.exists() else []
        if candidates:
            img_path = candidates[0]
            lbl_path = lbl_dir / img_path.with_suffix('.txt').name
            img = cv2.imread(str(img_path))
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            h, w = img.shape[:2]
            ax.imshow(img_rgb, cmap='gray')
            if lbl_path.exists():
                with open(lbl_path) as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) == 5:
                            cx, cy, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                            x1 = (cx - bw/2) * w
                            y1 = (cy - bh/2) * h
                            rect = patches.Rectangle((x1, y1), bw*w, bh*h,
                                                     linewidth=2, edgecolor=COLOR_MAP[cls_name], facecolor='none')
                            ax.add_patch(rect)
            color = COLOR_MAP[cls_name]
            ax.set_title(f'{zh_names_map[cls_name]}\n({cls_name})', fontsize=11, fontweight='bold')
            ax.axis('off')
            found = True
            break
    if not found:
        ax.text(0.5, 0.5, '无样本', ha='center', va='center', transform=ax.transAxes)
        ax.axis('off')

plt.tight_layout()
plt.savefig('../results/demo/sample_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ 已保存: results/demo/sample_visualization.png')

## 5. 目标框尺寸分布分析

In [ ]:
if bbox_sizes:
    widths = [s[0] for s in bbox_sizes]
    heights = [s[1] for s in bbox_sizes]
    areas = [w * h for w, h in bbox_sizes]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].hist(widths, bins=25, color='#4C72B0', edgecolor='white', alpha=0.85)
    axes[0].axvline(np.mean(widths), color='red', linestyle='--', label=f'均值={np.mean(widths):.1f}px')
    axes[0].set_title('目标框宽度分布', fontweight='bold')
    axes[0].set_xlabel('宽度 (px)')
    axes[0].legend()

    axes[1].hist(heights, bins=25, color='#DD8452', edgecolor='white', alpha=0.85)
    axes[1].axvline(np.mean(heights), color='red', linestyle='--', label=f'均值={np.mean(heights):.1f}px')
    axes[1].set_title('目标框高度分布', fontweight='bold')
    axes[1].set_xlabel('高度 (px)')
    axes[1].legend()

    axes[2].scatter([w for w,h in bbox_sizes], [h for w,h in bbox_sizes],
                    alpha=0.3, s=10, c='#55A868')
    axes[2].set_title('目标框宽高散点图', fontweight='bold')
    axes[2].set_xlabel('宽度 (px)')
    axes[2].set_ylabel('高度 (px)')

    plt.suptitle('目标框尺寸分布分析', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../results/demo/bbox_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'\n统计摘要:')
    print(f'  平均宽度: {np.mean(widths):.1f}px  |  平均高度: {np.mean(heights):.1f}px')
    print(f'  平均面积: {np.mean(areas):.1f}px²  |  总标注框数: {len(bbox_sizes)}')

## 6. 数据增强效果预览

In [ ]:
import albumentations as A

augment_pipeline = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.8),
    A.GaussNoise(var_limit=(10, 50), p=0.5),
    A.Rotate(limit=15, p=0.5),
    A.CLAHE(clip_limit=4.0, p=0.3),
])

# 找一张图来演示
sample_imgs = list((data_path / 'images' / 'train').glob('*.jpg'))
if sample_imgs:
    orig_img = cv2.cvtColor(cv2.imread(str(sample_imgs[0])), cv2.COLOR_BGR2RGB)

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    fig.suptitle('数据增强效果预览（训练时随机应用）', fontsize=14, fontweight='bold')

    axes[0, 0].imshow(orig_img)
    axes[0, 0].set_title('原图', fontweight='bold')
    axes[0, 0].axis('off')

    aug_labels = ['水平翻转', '亮度对比度', '高斯噪声', '随机旋转',
                  'CLAHE增强', '组合增强1', '组合增强2']
    single_augs = [
        A.Compose([A.HorizontalFlip(p=1.0)]),
        A.Compose([A.RandomBrightnessContrast(0.4, 0.4, p=1.0)]),
        A.Compose([A.GaussNoise(var_limit=(30, 80), p=1.0)]),
        A.Compose([A.Rotate(limit=20, p=1.0)]),
        A.Compose([A.CLAHE(clip_limit=6.0, p=1.0)]),
        augment_pipeline,
        augment_pipeline,
    ]

    for ax, (label, aug) in zip(list(axes.flat)[1:], zip(aug_labels, single_augs)):
        aug_img = aug(image=orig_img)['image']
        ax.imshow(aug_img)
        ax.set_title(label, fontsize=9, fontweight='bold')
        ax.axis('off')

    plt.tight_layout()
    plt.savefig('../results/demo/augmentation_preview.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✓ 已保存: results/demo/augmentation_preview.png')

## 总结

| 项目 | 数值 |
|------|------|
| 总样本数 | ~600 张（演示数据） |
| 类别数 | 6 类 |
| 类别平衡情况 | 基本均衡（各类约100张） |
| 图像尺寸 | 200×200 px |
| 数据增强策略 | 翻转 + 亮度/对比度 + 噪声 + 旋转 + CLAHE |

> **下一步**: 进入 `02_model_training.ipynb` 开始模型训练